# Grup C — NER dengan POS-tag (IndoBERT + POS Embedding)

Struktur notebook ini **identik dengan S1 baseline**, dengan 3 perubahan minimal:
1. `BertPosNER` — model custom (IndoBERT 768 + POS embedding 32 → concat 800 → Linear)
2. `df_to_dataset_for_model()` — tambah `pos_ids` feature (UPOS → integer, dialign per subword)
3. `filter_threshold()` — inference manual (bukan `pipeline()`) karena model punya input `pos_ids` tambahan

Semua fungsi evaluasi, output file (xlsx/csv), loop iterasi, dan hyperparameter **sama persis dengan S1**.

**Cara pakai:**
1. Jalankan `generate_pos_tags_colab.ipynb` → download zip → extract → upload `train.csv`, `test.csv`, `unlabelled.csv` ke `MyDrive/TA-Sirah/` (kolom `pos_tag` sudah terisi UPOS Stanza)
2. Runtime > Change runtime type > **GPU (T4)**
3. Run All

## Install dependency

In [ ]:
!pip install -q -U transformers accelerate seaborn seqeval

In [ ]:
from dataclasses import dataclass
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModel, AutoConfig,
    TrainingArguments, Trainer,
    DataCollatorForTokenClassification,
)
from transformers.modeling_outputs import TokenClassifierOutput
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
    precision_recall_fscore_support,
)
from shutil import rmtree
from tqdm import tqdm

import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import os

try:
    from seqeval.metrics import classification_report as seq_classification_report
    from seqeval.metrics import f1_score as seq_f1_score
    from seqeval.metrics import precision_score as seq_precision_score
    from seqeval.metrics import recall_score as seq_recall_score
    _HAS_SEQEVAL = True
except ImportError:
    _HAS_SEQEVAL = False
    print('[warn] seqeval tidak tersedia — jalankan `pip install seqeval` untuk metric entity-level')

## Load and prepare datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
_name = 'bert-pos'
_type = 'sirah-ner'
experiment_name = f'{_name}-{_type}'

dataset_dir = '/content/drive/MyDrive/TA-Sirah'
root_dir    = '/content/drive/MyDrive/TA-Sirah/output_pos_tag'
model_dir   = os.path.join(root_dir, 'models')
eval_dir    = os.path.join(root_dir, 'evaluation')
os.makedirs(model_dir, exist_ok=True)
os.makedirs(eval_dir,  exist_ok=True)

In [ ]:
df_train = pd.read_csv(os.path.join(dataset_dir, 'train.csv'))
df_train['token'] = df_train['token'].apply(str)
df_train['label'] = df_train['label'].apply(lambda x: x.replace('-', '_'))
df_train.loc[df_train[df_train.isna().any(axis=1)].index, 'token'] = 'nan'
if 'pos_tag' not in df_train.columns:
    print('[warn] pos_tag tidak ada — isi NN (jalankan generate_pos_tags_colab.ipynb dulu)')
    df_train['pos_tag'] = 'NN'

In [ ]:
import torch, os
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU            :', torch.cuda.get_device_name(0))
else:
    print('!! GPU tidak aktif. Runtime > Change runtime type > T4 GPU')

for f in ['train.csv', 'test.csv', 'unlabelled.csv']:
    fp = os.path.join(dataset_dir, f)
    print(f'{f:<16} ->', 'OK' if os.path.exists(fp) else 'MISSING', fp)

In [ ]:
label_list = sorted(df_train['label'].unique(), key=lambda name: (name[1:], name[0]))
id2label   = {i: label for i, label in enumerate(label_list)}
label2id   = {v: k for k, v in id2label.items()}

# POS vocabulary — UPOS Stanza + fallback 'NN' (placeholder data lama)
UPOS_LIST = [
    'PAD', 'UNK',
    'ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ',
    'NOUN', 'NUM', 'PART', 'PRON', 'PROPN', 'PUNCT',
    'SCONJ', 'SYM', 'VERB', 'X', 'NN',
]
pos2id  = {p: i for i, p in enumerate(UPOS_LIST)}
NUM_POS = len(pos2id)
POS_DIM = 32

print('labels  :', id2label)
print(f'POS vocab: {NUM_POS} tag')

### [BARU] BertPosNER

Model custom yang menggantikan `AutoModelForTokenClassification`.
Mengembalikan `TokenClassifierOutput` (sama) sehingga HF `Trainer` bekerja tanpa modifikasi.

**Save/load:** HF Trainer hanya tahu cara menyimpan state-dict mentah (bukan `from_pretrained`).
Karena arsitektur BERT selalu sama (`MODEL_NAME`), saat reload kita rebuild kerangka dari
config `MODEL_NAME` lalu `load_state_dict()`. Dua factory membantu ini:
- `new_pos_model_pretrained()` — BERT berbobot pretrained (untuk base / iterasi 0).
- `load_pos_model(dir)` — rebuild + load full state-dict (untuk inference & lanjutan self-training).

In [ ]:
class BertPosNER(nn.Module):
    """IndoBERT hidden (768) + POS embedding (32) -> concat (800) -> Dropout -> Linear -> num_labels."""
    def __init__(self, bert_module, num_labels, num_pos, pos_dim=32, dropout=0.1):
        super().__init__()
        self.bert       = bert_module
        self.pos_emb    = nn.Embedding(num_pos, pos_dim, padding_idx=0)  # 0=PAD
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(bert_module.config.hidden_size + pos_dim, num_labels)
        self.num_labels = num_labels

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None,
                pos_ids=None, labels=None, **kwargs):
        h      = self.bert(input_ids=input_ids,
                           attention_mask=attention_mask,
                           token_type_ids=token_type_ids).last_hidden_state  # (B, L, 768)
        p      = self.pos_emb(pos_ids if pos_ids is not None
                              else torch.zeros_like(input_ids))              # (B, L, 32)
        out    = self.dropout(torch.cat([h, p], dim=-1))                     # (B, L, 800)
        logits = self.classifier(out)                                         # (B, L, num_labels)
        loss   = None
        if labels is not None:
            loss = nn.CrossEntropyLoss(ignore_index=-100)(
                logits.view(-1, self.num_labels), labels.view(-1))
        return TokenClassifierOutput(loss=loss, logits=logits)


def new_pos_model_pretrained():
    """Model baru dengan BERT berbobot pretrained (untuk base / dari nol)."""
    bert = AutoModel.from_pretrained(MODEL_NAME)
    return BertPosNER(bert, len(label_list), NUM_POS, POS_DIM)


def load_pos_model(checkpoint_dir):
    """
    Rekonstruksi BertPosNER dari folder checkpoint hasil train_model().
    Arsitektur BERT selalu sama (MODEL_NAME) -> rebuild kerangka dari config,
    lalu isi dengan full state_dict (bert.* + pos_emb.* + classifier.*).
    """
    config = AutoConfig.from_pretrained(MODEL_NAME)
    bert   = AutoModel.from_config(config)               # arsitektur saja (bobot random)
    model  = BertPosNER(bert, len(label_list), NUM_POS, POS_DIM)
    state  = torch.load(os.path.join(checkpoint_dir, 'pytorch_model.bin'), map_location='cpu')
    model.load_state_dict(state)
    return model

### [BARU] DataCollatorForPosNER

Mewarisi `DataCollatorForTokenClassification` (padding input_ids/attention_mask/labels sama).
Tambahan: pad `pos_ids` dengan 0 (= tag PAD).

In [ ]:
@dataclass
class DataCollatorForPosNER(DataCollatorForTokenClassification):
    def __call__(self, features):
        # Pisahkan pos_ids sebelum parent collator (yang tidak mengenalinya)
        pos_ids_list = [f.pop('pos_ids') for f in features]
        batch        = super().__call__(features)  # parent handle input_ids / labels / dll

        # Pad pos_ids ke panjang sequence yang sama
        max_len = batch['input_ids'].shape[1]
        padded  = [p + [0] * (max_len - len(p)) for p in pos_ids_list]
        batch['pos_ids'] = torch.tensor(padded, dtype=torch.long)
        return batch

## Tokenize and split dataset into train and validation

In [ ]:
MODEL_NAME = 'indolem/indobert-base-uncased'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME, model_max_length=512)

_BIO_SCHEME = any(str(lab).startswith(('B_', 'I_')) for lab in id2label.values())


def tokenize_and_align_labels(examples):
    """Sama seperti S1, tambahan: align pos_ids per subword."""
    tokenized_inputs = tokenizer(examples['tokens'], truncation=True, is_split_into_words=True)

    labels_out  = []
    pos_ids_out = []
    for i, label in enumerate(examples['label']):
        word_ids      = tokenized_inputs.word_ids(batch_index=i)
        pos_raw       = examples['pos_ids'][i]   # integer per kata asli
        previous_word_idx = None
        label_ids     = []
        pos_aligned   = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
                pos_aligned.append(0)              # PAD untuk [CLS]/[SEP]
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
                pos_aligned.append(pos_raw[word_idx])
            else:
                label_ids.append(-100)             # subword kedua+ → ignore
                pos_aligned.append(pos_raw[word_idx])
            previous_word_idx = word_idx
        labels_out.append(label_ids)
        pos_ids_out.append(pos_aligned)

    tokenized_inputs['labels']  = labels_out
    tokenized_inputs['pos_ids'] = pos_ids_out
    return tokenized_inputs


def df_to_dataset_for_model(df: pd.DataFrame, val_text_ids: list | None = None):
    """
    Sama seperti S1 — hanya tambah kolom pos_ids di setiap row grouped.
    """
    df = df.copy(deep=True)
    df['label']   = df['label'].fillna('O')
    df['label']   = df['label'].apply(lambda x: label2id[x])
    if 'pos_tag' not in df.columns:
        df['pos_tag'] = 'NN'

    tmp_df          = df.groupby('text_id')['token'].apply(list).reset_index()
    tmp_df['label'] = df.groupby('text_id')['label'].apply(list).reset_index()['label']
    tmp_df['pos_ids'] = df.groupby('text_id')['pos_tag'].apply(
        lambda tags: [pos2id.get(str(t), pos2id['UNK']) for t in tags]
    ).reset_index()['pos_tag']
    tmp_df.columns = ['text_id', 'tokens', 'label', 'pos_ids']

    tmp_list = []
    for i in tmp_df.index:
        tmp_list.append({
            'text_id': tmp_df.loc[i, 'text_id'],
            'tokens':  tmp_df.loc[i, 'tokens'],
            'label':   tmp_df.loc[i, 'label'],
            'pos_ids': tmp_df.loc[i, 'pos_ids'],
        })

    dataset           = Dataset.from_list(tmp_list)
    tokenized_dataset = dataset.map(
        tokenize_and_align_labels,
        batched=True,
        remove_columns=[c for c in dataset.column_names if c != 'text_id'],
    )

    if val_text_ids is None:
        splits    = tokenized_dataset.train_test_split(test_size=0.2, seed=42)
        train_val = DatasetDict({'train': splits['train'], 'validation': splits['test']})
    else:
        val_set   = set(map(str, val_text_ids))
        is_val    = [str(x) in val_set for x in tokenized_dataset['text_id']]
        val_idx   = [i for i, v in enumerate(is_val) if v]
        trn_idx   = [i for i, v in enumerate(is_val) if not v]
        train_val = DatasetDict({
            'train':      tokenized_dataset.select(trn_idx).remove_columns(['text_id']),
            'validation': tokenized_dataset.select(val_idx).remove_columns(['text_id']),
        })

    return train_val


train_val = df_to_dataset_for_model(df_train)
print(train_val)

## Begin training base model

In [ ]:
# DataCollatorForPosNER mewarisi DataCollatorForTokenClassification
# — padding input_ids/labels sama seperti S1, tambahan pad pos_ids
data_collator = DataCollatorForPosNER(tokenizer=tokenizer)


def compute_metrics(pred):
    """Sama persis dengan S1 — token-level weighted F1 + seqeval entity-level."""
    labels      = pred.label_ids
    predictions = np.argmax(pred.predictions, axis=2)

    true_labels      = [[l for l, p in zip(label, prediction) if l != -100]
                        for label, prediction in zip(labels, predictions)]
    true_predictions = [[p for l, p in zip(label, prediction) if l != -100]
                        for label, prediction in zip(labels, predictions)]

    flat_true = [item for sub in true_labels      for item in sub]
    flat_pred = [item for sub in true_predictions for item in sub]

    precision, recall, f1, _ = precision_recall_fscore_support(
        flat_true, flat_pred, average='weighted', zero_division=0)

    out = {'precision': precision, 'recall': recall, 'f1': f1}

    if _HAS_SEQEVAL:
        true_lbl = [[id2label[l] for l in seq] for seq in true_labels]
        pred_lbl = [[id2label[p] for p in seq] for seq in true_predictions]
        try:
            out['seq_f1']        = seq_f1_score(true_lbl, pred_lbl)
            out['seq_precision'] = seq_precision_score(true_lbl, pred_lbl)
            out['seq_recall']    = seq_recall_score(true_lbl, pred_lbl)
        except Exception:
            pass
    return out


def train_model(model_name: str,
                train_dataset: Dataset,
                val_dataset: Dataset,
                model_output_path: str,
                num_train_epochs: int = 10):
    """
    Sama seperti S1.
    Perbedaan: model_init() return BertPosNER (bukan AutoModelForTokenClassification).
      - model_name = MODEL_NAME (hub id)  -> base: BERT pretrained, pos_emb/classifier baru.
      - model_name = path folder iterasi  -> lanjutan: load full state_dict (bobot terjaga).
    Setelah training, state_dict terbaik disimpan eksplisit ke `pytorch_model.bin`
    (independen versi/format Trainer) supaya load_pos_model() bisa membacanya.
    """
    training_args = TrainingArguments(
        output_dir=os.path.join(model_dir, '_trainer_tmp'),
        eval_strategy='epoch',
        save_strategy='epoch',
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=num_train_epochs,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model='f1',
        seed=42,
    )
    # Paksa simpan format .bin kalau versi transformers mendukung save_safetensors
    # (versi lama tak punya atribut ini -> default sudah .bin, jadi aman diabaikan).
    if hasattr(training_args, 'save_safetensors'):
        training_args.save_safetensors = False

    def model_init():
        if os.path.isdir(model_name):
            return load_pos_model(model_name)     # lanjutan iterasi: full weights terjaga
        return new_pos_model_pretrained()         # base: BERT pretrained dari hub

    trainer = Trainer(
        model_init=model_init,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    # Simpan state_dict terbaik secara EKSPLISIT -> pytorch_model.bin
    # (trainer.model sudah = model terbaik karena load_best_model_at_end=True)
    os.makedirs(model_output_path, exist_ok=True)
    torch.save(trainer.model.state_dict(),
               os.path.join(model_output_path, 'pytorch_model.bin'))

    rmtree(os.path.join(model_dir, '_trainer_tmp'), ignore_errors=True)

### Helper: extract entities (pengganti `extract_entities_from_result` S1)

Di S1, fungsi ini memetakan output `pipeline()` ke label per token asli.
Di sini, memetakan logits (sudah dalam bentuk array) ke label per token asli via `word_ids()`.

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def _load_model_for_inference(model_path: str) -> BertPosNER:
    """Load BertPosNER dari folder output train_model(), pindah ke DEVICE, mode eval."""
    return load_pos_model(model_path).to(DEVICE).eval()


def extract_entities_from_result(tokens, pos_strs, logits_np):
    """
    Pengganti `extract_entities_from_result(tokens, result)` dari S1.
    Input : logits_np (seq_len_with_special, num_labels) sudah include [CLS]/[SEP]
    Output: list label per token asli (len = len(tokens))
    """
    enc      = tokenizer(tokens, truncation=True, is_split_into_words=True)
    word_ids = enc.word_ids()
    pred_labels = []
    prev_w      = None
    for w, row in zip(word_ids, logits_np):
        if w is None or w == prev_w:
            prev_w = w
            continue
        pred_labels.append(id2label[row.argmax()])
        prev_w = w
    # FIX alignment: word yang terpotong truncation (chunk >512 subword)
    # tidak dapat prediksi -> samakan panjang dengan token asli (pad 'O').
    if len(pred_labels) < len(tokens):
        pred_labels += ['O'] * (len(tokens) - len(pred_labels))
    elif len(pred_labels) > len(tokens):
        pred_labels = pred_labels[:len(tokens)]
    return pred_labels

## Iterative pseudo-labelling (threshold=0.9)

In [ ]:
def filter_threshold(model_path: str,
                     df: pd.DataFrame,
                     threshold: float,
                     output_dir: str,
                     output_filename_prefix: str,
                     sampling_rate: float = 1.0,
                     min_entity_confidence: float | None = None,
                     aggregation_strategy: str = 'simple'):
    """
    Sama persis dengan S1: cache, above/below xlsx, retraining csv.
    Perbedaan:
      1. inference pakai forward pass manual karena model menerima pos_ids (tak bisa pipeline()).
      2. retraining csv ikut menyimpan kolom pos_tag, supaya saat retrain pseudo-label
         tetap memakai sinyal POS (bukan UNK).
    """
    cache_path = os.path.join(dataset_dir,
                              f'{output_filename_prefix}-above-{threshold}-retraining.csv')
    if os.path.exists(cache_path):
        tmp_df = pd.read_csv(cache_path)
        tmp_df['token'] = tmp_df['token'].apply(str)
        return tmp_df

    model    = _load_model_for_inference(model_path)
    text_ids = df['text_id'].unique().tolist()

    above = {'text_id': [], 'text': [], 'predicted_label': [], 'word': [], 'confidence': []}
    below = {'text_id': [], 'text': [], 'predicted_label': [], 'word': [], 'confidence': []}
    predicted_above_rows = []   # (text_id, tokens, pred_labels, pos_strs, avg)

    for text_id in tqdm(text_ids):
        grp      = df[df['text_id'] == text_id]
        tokens   = grp['token'].astype(str).tolist()
        pos_strs = grp['pos_tag'].astype(str).tolist() if 'pos_tag' in grp.columns else ['NN'] * len(tokens)
        text     = ' '.join(tokens)

        # ── Inference (pengganti pipeline) ────────────────────────────────
        enc         = tokenizer(tokens, truncation=True, is_split_into_words=True, return_tensors='pt')
        pos_raw     = [pos2id.get(p, pos2id['UNK']) for p in pos_strs]
        pos_aligned = [0 if w is None else pos_raw[w] for w in enc.word_ids()]
        pos_tensor  = torch.tensor([pos_aligned], dtype=torch.long)

        with torch.no_grad():
            out   = model(input_ids=enc['input_ids'].to(DEVICE),
                          attention_mask=enc['attention_mask'].to(DEVICE),
                          pos_ids=pos_tensor.to(DEVICE))
        probs = torch.softmax(out.logits[0], dim=-1).cpu().numpy()  # (seq_len, num_labels)
        # ─────────────────────────────────────────────────────────────────

        pred_labels = extract_entities_from_result(tokens, pos_strs, probs)

        # Konfidence entitas (bukan O) — analog avg score entity-group di pipeline
        prev_w = None
        entity_scores = []
        for w, row in zip(enc.word_ids(), probs):
            if w is None or w == prev_w:
                prev_w = w
                continue
            lab = id2label[row.argmax()]
            if lab != 'O':
                entity_scores.append({'word': tokens[w], 'score': float(row.max()), 'entity_group': lab})
            prev_w = w

        if len(entity_scores) == 0:
            below['text_id'].append(text_id)
            below['text'].append(text)
            below['predicted_label'].append('-')
            below['word'].append('-')
            below['confidence'].append(0)
            continue

        scores = [e['score'] for e in entity_scores]
        avg    = sum(scores) / len(scores)
        passes_min = (min_entity_confidence is None) or (min(scores) >= min_entity_confidence)

        if avg >= threshold and passes_min:
            predicted_above_rows.append((text_id, tokens, pred_labels, pos_strs, avg))
            for e in entity_scores:
                above['text_id'].append(text_id)
                above['text'].append(text)
                above['predicted_label'].append(e['entity_group'])
                above['word'].append(e['word'])
                above['confidence'].append(e['score'])
        else:
            for e in entity_scores:
                below['text_id'].append(text_id)
                below['text'].append(text)
                below['predicted_label'].append(e['entity_group'])
                below['word'].append(e['word'])
                below['confidence'].append(e['score'])

    if 0 < sampling_rate < 1.0 and predicted_above_rows:
        predicted_above_rows.sort(key=lambda r: r[4], reverse=True)   # r[4] = avg
        keep = max(1, int(round(len(predicted_above_rows) * sampling_rate)))
        predicted_above_rows = predicted_above_rows[:keep]

    above_df = pd.DataFrame(above)
    below_df = pd.DataFrame(below)
    above_df.to_excel(os.path.join(output_dir, f'{output_filename_prefix}-above-{threshold}.xlsx'))
    below_df.to_excel(os.path.join(output_dir, f'{output_filename_prefix}-below-{threshold}.xlsx'))

    n_above = len({r[0] for r in predicted_above_rows})
    print(f'Above {threshold}: {n_above} sentences (sampling_rate={sampling_rate})')
    print(f'Below {threshold}: {below_df["text_id"].nunique() if len(below_df) else 0} sentences')

    # Retraining csv — ikut pos_tag agar pseudo-label tetap berfitur POS saat retrain
    retrain_rows = []
    for text_id, toks, labs, poss, _ in predicted_above_rows:
        for t, lab, pos in zip(toks, labs, poss):
            retrain_rows.append({'text_id': text_id, 'token': t, 'label': lab, 'pos_tag': pos})
    retrain_df = pd.DataFrame(retrain_rows, columns=['text_id', 'token', 'label', 'pos_tag'])
    retrain_df.to_csv(cache_path, index=False)
    return retrain_df


def reconstruct_unlabelled_from_below(below_xlsx_path: str) -> pd.DataFrame:
    """
    Ambil chunk di-bawah-threshold untuk iterasi berikutnya.
    Beda dari S1: alih-alih split ulang teks, kita pilih baris ASLI dari df_unlabelled
    (token + pos_tag utuh) berdasarkan text_id yang masuk below — supaya POS tidak hilang.
    """
    df = pd.read_excel(below_xlsx_path)
    if len(df) == 0 or 'text_id' not in df.columns:
        return pd.DataFrame(columns=['text_id', 'token', 'pos_tag'])
    below_ids = set(df['text_id'].unique())
    return df_unlabelled[df_unlabelled['text_id'].isin(below_ids)].copy()

In [ ]:
N_ITERATIONS    = 6
THRESHOLD       = 0.9
SAMPLING_RATE   = 1.0
MIN_ENTITY_CONF = None
MIN_NEW_SAMPLES = 0
AGG_STRATEGY    = 'simple'

import random as _random
_seed_ids    = sorted(df_train['text_id'].unique().tolist())
_rng         = _random.Random(42)
_rng.shuffle(_seed_ids)
_n_val       = max(1, int(round(len(_seed_ids) * 0.2)))
VAL_TEXT_IDS = _seed_ids[:_n_val]
print(f'[val split] {len(VAL_TEXT_IDS)} / {len(_seed_ids)} seed text_ids reserved for validation')

df_unlabelled = pd.read_csv(os.path.join(dataset_dir, 'unlabelled.csv'))
df_unlabelled['token'] = df_unlabelled['token'].apply(str)
if 'pos_tag' not in df_unlabelled.columns:
    df_unlabelled['pos_tag'] = 'NN'

# Bootstrap: train base model
train_val_base = df_to_dataset_for_model(df_train, val_text_ids=VAL_TEXT_IDS)
print(train_val_base)

train_model(
    model_name=MODEL_NAME,
    train_dataset=train_val_base['train'],
    val_dataset=train_val_base['validation'],
    model_output_path=os.path.join(model_dir, f'{experiment_name}-base'),
)

# Iterative loop
pseudo_dfs         = []
iter_log           = []
current_unlabelled = df_unlabelled
current_model_name = f'{experiment_name}-base'

for i in range(1, N_ITERATIONS + 1):
    if len(current_unlabelled) == 0:
        print(f'[iter {i}] no unlabelled chunks left -> STOP (self-training converged at iter {i-1})')
        break

    prev_model_path = os.path.join(model_dir, current_model_name)
    prefix = f'{experiment_name}' if i == 1 else f'{experiment_name}-iterative-{i}'

    above_df = filter_threshold(
        model_path=prev_model_path,
        df=current_unlabelled,
        threshold=THRESHOLD,
        output_dir=eval_dir,
        output_filename_prefix=prefix,
        sampling_rate=SAMPLING_RATE,
        min_entity_confidence=MIN_ENTITY_CONF,
        aggregation_strategy=AGG_STRATEGY,
    )

    n_above = int(above_df['text_id'].nunique()) if len(above_df) else 0
    print(f'[iter {i}] above threshold: {n_above} sentences')
    iter_log.append({'iter': i, 'n_above': n_above})

    if MIN_NEW_SAMPLES > 0 and n_above < MIN_NEW_SAMPLES:
        print(f'[iter {i}] early-stop: {n_above} < MIN_NEW_SAMPLES={MIN_NEW_SAMPLES}')
        break

    pseudo_dfs.append(above_df)

    if i < N_ITERATIONS:
        new_model_name = f'{experiment_name}-0.9-iteration-{i+1}'
        combined       = pd.concat([df_train] + pseudo_dfs, ignore_index=True)
        train_val_iter = df_to_dataset_for_model(combined, val_text_ids=VAL_TEXT_IDS)
        print(train_val_iter)

        train_model(
            model_name=prev_model_path,
            train_dataset=train_val_iter['train'],
            val_dataset=train_val_iter['validation'],
            model_output_path=os.path.join(model_dir, new_model_name),
        )
        current_model_name = new_model_name

        below_path         = os.path.join(eval_dir, f'{prefix}-below-{THRESHOLD}.xlsx')
        current_unlabelled = reconstruct_unlabelled_from_below(below_path)

pd.DataFrame(iter_log).to_csv(os.path.join(eval_dir, 'iteration_log.csv'), index=False)
print('\nIteration log:')
print(pd.DataFrame(iter_log).to_string(index=False))

final_iter  = iter_log[-1]['iter']
above_09_df = pseudo_dfs[0] if pseudo_dfs else pd.DataFrame()

## Evaluate model performance

In [ ]:
def get_predicted_label_on_test_dataset(model_path: str, df: pd.DataFrame):
    """Sama seperti S1, tapi inference manual (bukan pipeline)."""
    model     = _load_model_for_inference(model_path)
    predicted = []
    text_ids  = df['text_id'].unique().tolist()

    for text_id in tqdm(text_ids):
        tokens   = df[df['text_id'] == text_id]['token'].astype(str).tolist()
        pos_strs = df[df['text_id'] == text_id]['pos_tag'].astype(str).tolist() \
                   if 'pos_tag' in df.columns else ['NN'] * len(tokens)

        enc         = tokenizer(tokens, truncation=True, is_split_into_words=True, return_tensors='pt')
        pos_raw     = [pos2id.get(p, pos2id['UNK']) for p in pos_strs]
        pos_aligned = [0 if w is None else pos_raw[w] for w in enc.word_ids()]
        pos_tensor  = torch.tensor([pos_aligned], dtype=torch.long)

        with torch.no_grad():
            out   = model(input_ids=enc['input_ids'].to(DEVICE),
                          attention_mask=enc['attention_mask'].to(DEVICE),
                          pos_ids=pos_tensor.to(DEVICE))
        probs = torch.softmax(out.logits[0], dim=-1).cpu().numpy()
        predicted.extend(extract_entities_from_result(tokens, pos_strs, probs))

    df['predicted_label'] = predicted


def get_overall_performance(df: pd.DataFrame):
    total   = len(df)
    correct = (df['label'] == df['predicted_label']).sum()
    print('Accuracy:', correct / total)
    y_true    = df['label']
    y_pred    = df['predicted_label']
    precision = precision_score(y_true, y_pred, average='weighted', labels=label_list)
    recall    = recall_score(y_true, y_pred, average='weighted', labels=label_list)
    f1        = f1_score(y_true, y_pred, average='weighted', labels=label_list)
    print('Precision:', precision)
    print('Recall:',    recall)
    print('F1 Score:',  f1)


def get_individual_label_score(df: pd.DataFrame):
    y_true  = df['label']
    y_pred  = df['predicted_label']
    report  = classification_report(y_true, y_pred, labels=label_list, output_dict=True)
    return pd.DataFrame(report).transpose()


def get_confusion_matrix(df: pd.DataFrame):
    y_true = df['label']
    y_pred = df['predicted_label']
    cm     = confusion_matrix(y_true, y_pred, labels=label_list)
    ax     = plt.subplot()
    sns.heatmap(cm, annot=True, fmt='g', ax=ax, cmap='Oranges', vmin=0, vmax=160)
    ax.set_xlabel('Predicted labels')
    ax.set_ylabel('True labels')
    ax.set_title('Confusion Matrix')
    plt.xticks(rotation=270)
    plt.yticks(rotation=0)
    ax.xaxis.set_ticklabels(label_list)
    ax.yaxis.set_ticklabels(label_list)


def get_misclassified_report(df: pd.DataFrame, output_path: str):
    d = {'text_id': [], 'token': [], 'true_label': [], 'pred_label': [], 'text': []}
    for i in tqdm(df.index):
        if df.loc[i, 'label'] != df.loc[i, 'predicted_label']:
            text_id = df.loc[i, 'text_id']
            d['token'].append(df.loc[i, 'token'])
            d['text_id'].append(text_id)
            d['true_label'].append(df.loc[i, 'label'])
            d['pred_label'].append(df.loc[i, 'predicted_label'])
            d['text'].append(' '.join(df[df['text_id'] == text_id]['token'].tolist()))
    pd.DataFrame(d).to_excel(output_path)


def get_correct_and_incorrect_classification(df: pd.DataFrame, output_prefix: str):
    correct   = {'text_id': [], 'token': [], 'true_label': [], 'pred_label': [], 'text': []}
    incorrect = {'text_id': [], 'token': [], 'true_label': [], 'pred_label': [], 'text': []}
    for i in tqdm(df.index):
        text_id = df.loc[i, 'text_id']
        text    = ' '.join(df[df['text_id'] == text_id]['token'].tolist())
        d       = correct if df.loc[i, 'label'] == df.loc[i, 'predicted_label'] else incorrect
        d['token'].append(df.loc[i, 'token'])
        d['text_id'].append(text_id)
        d['true_label'].append(df.loc[i, 'label'])
        d['pred_label'].append(df.loc[i, 'predicted_label'])
        d['text'].append(text)
    pd.DataFrame(correct).to_excel(os.path.join(eval_dir, f'{output_prefix}-correct.xlsx'))
    pd.DataFrame(incorrect).to_excel(os.path.join(eval_dir, f'{output_prefix}-incorrect.xlsx'))

In [ ]:
df_test = pd.read_csv(os.path.join(dataset_dir, 'test.csv'))
df_test['label'] = df_test['label'].apply(lambda x: x.replace('-', '_'))
if 'pos_tag' not in df_test.columns:
    df_test['pos_tag'] = 'NN'

get_predicted_label_on_test_dataset(
    model_path=os.path.join(model_dir, f'{experiment_name}-0.9-iteration-{final_iter}'),
    df=df_test,
)

In [ ]:
get_overall_performance(df_test)
get_individual_label_score(df_test)

In [ ]:
get_confusion_matrix(df_test)

In [ ]:
get_misclassified_report(
    df_test,
    os.path.join(eval_dir, f'{experiment_name}-confidence-{THRESHOLD}-misclassified.xlsx')
)

In [ ]:
get_correct_and_incorrect_classification(df_test, f'{experiment_name}-iterative-{final_iter}')

In [ ]:
# Evaluasi base model (sebelum self-training) — untuk lihat gain iterasi
df_test_base = pd.read_csv(os.path.join(dataset_dir, 'test.csv'))
df_test_base['label'] = df_test_base['label'].apply(lambda x: x.replace('-', '_'))
if 'pos_tag' not in df_test_base.columns:
    df_test_base['pos_tag'] = 'NN'

get_predicted_label_on_test_dataset(
    model_path=os.path.join(model_dir, f'{experiment_name}-base'),
    df=df_test_base,
)
get_overall_performance(df_test_base)
get_individual_label_score(df_test_base)